# Setup

In [ ]:
import datasets
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

raw_dataset = datasets.load_dataset("sookiemonster/asrs-narratives")

utils.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [ ]:
train_ds = raw_dataset['train']
valid_ds = raw_dataset['validation']
test_ds = raw_dataset['test']

labels = train_ds.features['label'].names

id_to_label = { idx : label for idx, label in enumerate(labels) }
label_to_id = { label : idx for idx, label in id_to_label.items() }

In [ ]:
def get_mapped_ds(ds, prefix: str):
  res = ds.map(lambda x: {"text": prefix + x["text"]})
  return res

classification_train_ds = get_mapped_ds(train_ds, "classification: ")
clustering_train_ds = get_mapped_ds(train_ds, "clustering: ")

Map:   0%|          | 0/10360 [00:00<?, ? examples/s]

Map:   0%|          | 0/10360 [00:00<?, ? examples/s]

# Inference w/ Nomic AI


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "nomic-ai/modernbert-embed-base"
model = SentenceTransformer(MODEL_ID)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

W0320 01:10:26.166000 12516 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


In [ ]:
pd.Series(train_ds['label']).map(id_to_label).value_counts()

,count
humanfactors,3539
aircraft,3474
procedure,1016
ambiguous,835
weather,346
airport,256
environment-nonweatherrelated,250
chartorpublication,141
airspacestructure,135
atcequipment/navfacility/buildings,133


In [ ]:
classification_embeddings = model.encode(
    classification_train_ds['text'],
    batch_size=1,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/10360 [00:00<?, ?it/s]

In [ ]:
np.save("nomic-modernbert-clf-embeddings.npy", classification_embeddings)

In [ ]:
embed = classification_embeddings.copy()

In [ ]:
from umap import UMAP

um = UMAP(n_neighbors=62, min_dist=0.1, metric='cosine', random_state=42)
pred = um.fit_transform(embed)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
df = pd.DataFrame({
    'x': pred[:, 0],
    'y': pred[:, 1],
    'problem': train_ds['label']
})

df.problem = df.problem.map(id_to_label)
df

,x,y,problem
0,-0.779604,6.481331,weather
1,8.915415,8.192430,humanfactors
2,6.145708,9.020451,humanfactors
3,3.465748,8.258118,aircraft
4,4.489516,8.260482,aircraft
...,...,...,...
10355,-0.012055,9.117291,procedure
10356,0.767816,8.813838,ambiguous
10357,4.115117,8.164927,aircraft
10358,2.293866,6.193784,ambiguous


In [ ]:
import plotly.express as px
fig = px.scatter(x=pred[:,0], y=pred[:,1], color=df.problem)
fig.show()

In [ ]:
from collections import defaultdict

isolate = ['humanfactors', 'procedure', 'aircraft', 'atcequipment/navfacility/buildings', 'airport', 'airspacestructure']
groupings = defaultdict(lambda : 'grouped', {
    m : m for m in isolate
})

fig = px.scatter(x=pred[:,0], y=pred[:,1], color=df.problem.map(groupings))
fig.show()

In [ ]:
df.problem.map(groupings).value_counts()

,count
problem,
humanfactors,3539
aircraft,3474
grouped,1807
procedure,1016
airport,256
airspacestructure,135
atcequipment/navfacility/buildings,133


# Fine-Tuning

In [ ]:
# Form groups which have enough instances/show separability

from datasets import ClassLabel
from collections import defaultdict

def group_labels(ds):
  FILL_GROUP = 'grouped'
  groups = ['humanfactors', 'procedure', 'aircraft', 'atcequipment/navfacility/buildings', 'airport', 'airspacestructure', FILL_GROUP]
  groups.sort()

  mapping = defaultdict(lambda : FILL_GROUP, {
      m : m for m in groups
  })


  base_id_to_group = { idx: mapping[val] for idx, val in id_to_label.items()}
  res = ds.map(lambda x: {"group": base_id_to_group[x['label']]})

  new_features = res.features.copy()
  new_features["group"] = ClassLabel(names=groups)

  res = res.cast(new_features)
  return res

def format_for_triplet_classification_training(ds):
  c = get_mapped_ds(ds, "classification: ")
  c = group_labels(c)
  c = c.remove_columns(["label", "acn"]).rename_column('text', 'sentence').rename_column('group', 'label')
  return c



# Setup datasets
grouped_train_ds = format_for_triplet_classification_training(train_ds)
grouped_eval_ds = format_for_triplet_classification_training(valid_ds)

# Setup mappings
groups = grouped_train_ds.features['label'].names

id_to_group = { idx : label for idx, label in enumerate(groups) }
group_to_id = { label : idx for idx, label in id_to_group.items() }

In [ ]:
pd.Series(grouped_train_ds['label']).value_counts(normalize=True)

,proportion
5,0.341602
0,0.335328
4,0.174421
6,0.098069
1,0.024710
2,0.013031
3,0.012838


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "nomic-ai/modernbert-embed-base"
model = SentenceTransformer(MODEL_ID)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.losses import BatchHardTripletLossDistanceFunction
from sentence_transformers import training_args

loss = losses.BatchSemiHardTripletLoss(
    model=model,
    distance_metric=BatchHardTripletLossDistanceFunction.cosine_distance
    #margin <-- use default
)

args = SentenceTransformerTrainingArguments(
    output_dir="models/nomic-modernbert-asrs",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    warmup_steps=200,
    fp16=True,
    gradient_accumulation_steps=32,
    batch_sampler=training_args.BatchSamplers.GROUP_BY_LABEL,
    # eval_strategy='epoch',
    save_strategy='steps',
    save_steps=300,
    report_to="none",
    logging_steps=50,
    push_to_hub=True,
    hub_model_id="sookiemonster/asrs-modernbert-nomic"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=grouped_train_ds,
    loss=loss,
)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
trainer.train()

Step,Training Loss
50,5.002147
100,4.875672
150,4.506331
200,4.422792
250,4.112949
300,4.459303


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=324, training_loss=4.500045352511936, metrics={'train_runtime': 999.3507, 'train_samples_per_second': 41.467, 'train_steps_per_second': 0.324, 'total_flos': 0.0, 'train_loss': 4.500045352511936, 'epoch': 4.0})

In [ ]:
import pandas as pd

# Extract the list of dictionaries containing your logs
log_history = trainer.state.log_history

# Convert to a readable dataframe
df = pd.DataFrame(log_history)

# Drop empty columns (HF logs 'loss' and 'eval_loss' on different steps, leaving NaNs)
df_clean = df.dropna(axis=1, how='all')
df_clean

,loss,grad_norm,learning_rate,epoch,step,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,5.002147,0.158636,0.000005,0.621842,50,NaN,NaN,NaN,NaN,NaN
1,4.875672,4.199338,0.000010,1.236300,100,NaN,NaN,NaN,NaN,NaN
2,4.506331,4.139358,0.000015,1.858142,150,NaN,NaN,NaN,NaN,NaN
3,4.422792,11.447140,0.000020,2.472600,200,NaN,NaN,NaN,NaN,NaN
4,4.112949,6.320327,0.000012,3.087058,250,NaN,NaN,NaN,NaN,NaN
5,4.459303,10.059128,0.000004,3.708900,300,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,4.000000,324,999.3507,41.467,0.324,0.0,4.500045


# Inference

In [ ]:
fitted = SentenceTransformer("sookiemonster/asrs-modernbert-nomic")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/596M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
classification_embeddings = fitted.encode(
    classification_train_ds['text'],
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/324 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


In [ ]:
np.save("fitted-clf-embeddings.npy", classification_embeddings)

In [ ]:
embed = classification_embeddings.copy()

# Clustering


In [ ]:
embed  = np.load('fitted-clf-embeddings.npy')

In [ ]:
from umap import UMAP

um = UMAP(n_neighbors=62, min_dist=0.1, metric='cosine', random_state=42)
pred = um.fit_transform(embed)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
df = pd.DataFrame({
    'x': pred[:, 0],
    'y': pred[:, 1],
    'problem': train_ds['label']
})

df.problem = df.problem.map(id_to_label)
df

,x,y,problem
0,-0.779604,6.481331,weather
1,8.915415,8.192430,humanfactors
2,6.145708,9.020451,humanfactors
3,3.465748,8.258118,aircraft
4,4.489516,8.260482,aircraft
...,...,...,...
10355,-0.012055,9.117291,procedure
10356,0.767816,8.813838,ambiguous
10357,4.115117,8.164927,aircraft
10358,2.293866,6.193784,ambiguous


In [ ]:
import plotly.express as px
fig = px.scatter(x=pred[:,0], y=pred[:,1], color=df.problem)
fig.show()

In [ ]:
from collections import defaultdict

isolate = ['humanfactors', 'procedure', 'aircraft', 'atcequipment/navfacility/buildings', 'airport', 'airspacestructure']
groupings = defaultdict(lambda : 'grouped', {
    m : m for m in isolate
})

fig = px.scatter(x=pred[:,0], y=pred[:,1], color=df.problem.map(groupings))
fig.show()

In [ ]:
groups = train_ds.features['label'].names

In [ ]:
import plotly.express as px

fig = px.scatter(x=pred[:,0], y=pred[:,1], color=pd.Series(train_ds['label']).map(lambda x : groups[x]))
fig.show()

In [ ]:
from umap import UMAP

um = UMAP(n_neighbors=10, min_dist=0.1, metric='cosine', random_state=42)
proj = um.fit_transform(embed)
fig = px.scatter(x=proj[:,0], y=proj[:,1], color=pd.Series(train_ds['label']).map(lambda x : groups[x]))
fig.show()

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:
ll =pd.Series(train_ds['label']).map(lambda x : groups[x])
ll

,0
0,weather
1,humanfactors
2,humanfactors
3,aircraft
4,aircraft
...,...
10355,procedure
10356,ambiguous
10357,aircraft
10358,ambiguous


In [ ]:
ll = (ll == 'aircraft')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

KNN = KNeighborsClassifier()
pp = cross_val_predict(KNN, pred, ll)
print(classification_report(ll, pp))

              precision    recall  f1-score   support

       False       0.93      0.92      0.92      6886
        True       0.84      0.86      0.85      3474

    accuracy                           0.90     10360
   macro avg       0.88      0.89      0.88     10360
weighted avg       0.90      0.90      0.90     10360

